# Fillweight Analysis: The Power of ProcessBehavior

This notebook demonstrates the power and flexibility of ProcessBehavior for analyzing fillweight data across multiple stratification strategies.

We'll analyze the same fillweight dataset three different ways:
1. **By Lane**: Understanding machine-to-machine variation
2. **By Phase**: Understanding temporal process changes
3. **By Lane × Phase**: Full stratification for deepest insights

Each analysis takes just 2-3 lines of code, yet provides:
- Automatic SDS (Sampling Design State) detection
- Appropriate control charts based on data structure
- Signal detection with WECO rules
- Group statistics and diagnostics

**Key Feature**: With the new type conversion and natural sorting, charts display correctly ordered labels regardless of whether your data uses numeric or string identifiers.

In [1]:
import pandas as pd
import sys
sys.path.insert(0, '..')

from processbehavior import ProcessDataFrame

# Load fillweight data
df = pd.read_csv('../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv')

print(f"Dataset: {len(df)} observations")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head(10)

Dataset: 800 observations
Columns: ['pull', 'lane', 'phase', 'fill_weight']

First few rows:


,pull,lane,phase,fill_weight
0,1,1,1,236.93
1,1,1,2,237.39
2,1,2,1,236.30
3,1,2,2,241.35
4,1,3,1,236.09
5,1,3,2,232.30
6,1,4,1,235.81
7,1,4,2,241.89
8,2,1,1,239.67
9,2,1,2,236.39


## Data Overview

Let's understand the structure of our fillweight data:

In [ ]:
print("Data Structure:")
print(f"  Lanes: {sorted(df['lane'].unique())}")
print(f"  Phases: {sorted(df['phase'].unique())}")
print(f"  Pull range: {df['pull'].min()} to {df['pull'].max()}")
print(f"  Fillweight range: {df['fill_weight'].min():.2f} to {df['fill_weight'].max():.2f}")

# Check data types (before ProcessBehavior type conversion)
print(f"\nOriginal Data Types:")
print(f"  lane: {df['lane'].dtype}")
print(f"  phase: {df['phase'].dtype}")
print(f"  pull: {df['pull'].dtype}")

# Check data quality
print(f"\nData Quality:")
print(f"  Total rows in file: {len(df)}")
print(f"  Rows with missing values: {df.isna().any(axis=1).sum()}")
print(f"  Clean rows (will be used in analysis): {len(df.dropna())}")

---

# Analysis 1: By Lane

**Question**: Is there variation between filling machines/lanes?

**Strategy**: Group by lane, track over time (pull number)

This is the classic "machine comparison" analysis in manufacturing.

In [3]:
# Create ProcessDataFrame and analyze by lane
pdf = ProcessDataFrame(df)

analysis_by_lane = pdf.analyze(
    response_var='fill_weight',
    grouping_vars=['lane'],
    time_var='pull'
)

result_by_lane = analysis_by_lane.calculate()


PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 1: Full Factorial with Complete Replication
   All factor × time cells have n ≥ 2 observations (best case for analysis)
   Replication: Full

📈 Available charts: Xbar, S, Imr
   Selected: Xbar and S Charts (Subgroup Mean and Variation) (recommended)

📋 Data Configuration:
   Response: fill_weight
   Time: pull
   Grouping: lane

✨ Analysis Capabilities:
   • VAS residuals: R1, R2, R3, R4, R5 (R2 method: exact)
   • Main effects: Yes
   • Interactions: Yes




/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([spec.rsg_var_name], dropna=False)[spec.response_var]
/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:307: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grid_cells = df.groupby([spec.rsg_var_name, spec.time_var], dropna=False).size()
/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed

### Sampling Design State (SDS)

ProcessBehavior automatically detects the data structure:

In [4]:
sds = result_by_lane.summary['sds']
print(f"Detected SDS: {sds}")
print(f"\nSDS Interpretation:")
if sds == 1:
    print("  SDS 1: Full Replication")
    print("  - Multiple observations per (lane × time) combination")
    print("  - Both Xbar and S charts available")
    print("  - Can estimate within-subgroup variation")
elif sds == 2:
    print("  SDS 2: No Replication")
    print("  - Single observation per (lane × time) combination")
    print("  - Only Xbar chart available (no S chart)")
print(f"\nAvailable Charts: {list(result_by_lane.charts.keys())}")

Detected SDS: 1

SDS Interpretation:
  SDS 1: Full Replication
  - Multiple observations per (lane × time) combination
  - Both Xbar and S charts available
  - Can estimate within-subgroup variation

Available Charts: ['Xbar', 'Sbar']


### Group Statistics

Summary statistics for each lane:

In [ ]:
# Get group-level statistics
if 'Xbar' in result_by_lane.charts:
    # Combine Xbar and Sbar data for complete statistics
    xbar_data = result_by_lane.charts['Xbar']['data'][['rsg', 'xbar', 'lcl', 'ucl']].copy()
    sbar_data = result_by_lane.charts['Sbar']['data'][['rsg', 's']].copy() if 'Sbar' in result_by_lane.charts else None
    
    if sbar_data is not None:
        lane_stats = xbar_data.merge(sbar_data, on='rsg')
    else:
        lane_stats = xbar_data
    
    # Add observation counts from ANALYSIS dataset (after data cleaning)
    lane_counts = result_by_lane.dataset.groupby('rsg').size().reset_index(name='n')
    lane_stats = lane_stats.merge(lane_counts, on='rsg')
    
    # Sort and display
    lane_stats = lane_stats.sort_values('rsg')
    print("Lane Statistics:")
    print(lane_stats.to_string(index=False))
    
    print(f"\nTotal observations by lane (after data cleaning):")
    for _, row in lane_counts.iterrows():
        print(f"  Lane {row['rsg']}: {row['n']} observations")
    print(f"Total: {lane_counts['n'].sum()} observations")

### Signal Detection

Automatic detection of out-of-control conditions:

In [ ]:
signals = result_by_lane.get_signals()
if not signals.empty:
    print(f"Signals Detected: {len(signals)} signal(s)\n")
    for i, row in signals.iterrows():
        print(f"Signal #{i+1}:")
        print(f"  Chart: {row['chart']}")
        print(f"  Group: {row['rsg']}")
        print(f"  Beyond limits: {row['beyond_limits']}")
        if row['chart'] == 'Xbar':
            print(f"  Xbar value: {row['xbar']:.3f}, Center: {row['center']:.3f}")
            print(f"  Limits: [{row['lcl']:.3f}, {row['ucl']:.3f}]")
        elif row['chart'] == 'Sbar':
            print(f"  S value: {row['s']:.3f}, Center: {row['center']:.3f}")
            print(f"  Limits: [{row['lcl']:.3f}, {row['ucl']:.3f}]")
        print()
else:
    print("✓ No signals detected - Process appears stable by lane")

### Key Insight: Lane Analysis

This analysis answers: "Do different filling lanes produce consistently different results?"

In [ ]:
signals_by_lane = result_by_lane.get_signals()
print("\n" + "="*60)
print("LANE ANALYSIS SUMMARY")
print("="*60)
print(f"Lanes analyzed: {lane_stats['rsg'].nunique()}")
print(f"Observations per lane: {lane_stats['n'].mean():.0f} (average)")
print(f"Overall mean fillweight: {lane_stats['xbar'].mean():.3f}")
print(f"Lane-to-lane variation: {lane_stats['xbar'].std():.3f}")
print(f"Signals detected: {len(signals_by_lane)}")

---

# Analysis 2: By Phase

**Question**: Did the process change over time (between phases)?

**Strategy**: Group by phase, track over time (pull number)

This reveals whether process improvements or changes had an effect.

In [8]:
# Analyze by phase
analysis_by_phase = pdf.analyze(
    response_var='fill_weight',
    grouping_vars=['phase'],
    time_var='pull'
)

result_by_phase = analysis_by_phase.calculate()

print(f"Detected SDS: {result_by_phase.summary['sds']}")
print(f"Available Charts: {list(result_by_phase.charts.keys())}")


PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 1: Full Factorial with Complete Replication
   All factor × time cells have n ≥ 2 observations (best case for analysis)
   Replication: Full

📈 Available charts: Xbar, S, Imr
   Selected: Xbar and S Charts (Subgroup Mean and Variation) (recommended)

📋 Data Configuration:
   Response: fill_weight
   Time: pull
   Grouping: phase

✨ Analysis Capabilities:
   • VAS residuals: R1, R2, R3, R4, R5 (R2 method: exact)
   • Main effects: Yes
   • Interactions: Yes


Detected SDS: 1
Available Charts: ['Xbar', 'Sbar']


/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([spec.rsg_var_name], dropna=False)[spec.response_var]
/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:307: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grid_cells = df.groupby([spec.rsg_var_name, spec.time_var], dropna=False).size()
/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed

### Phase Statistics

In [ ]:
if 'Xbar' in result_by_phase.charts:
    # Combine Xbar and Sbar data
    xbar_data = result_by_phase.charts['Xbar']['data'][['rsg', 'xbar', 'lcl', 'ucl']].copy()
    sbar_data = result_by_phase.charts['Sbar']['data'][['rsg', 's']].copy() if 'Sbar' in result_by_phase.charts else None
    
    if sbar_data is not None:
        phase_stats = xbar_data.merge(sbar_data, on='rsg')
    else:
        phase_stats = xbar_data
    
    # Add observation counts from ANALYSIS dataset
    phase_counts = result_by_phase.dataset.groupby('rsg').size().reset_index(name='n')
    phase_stats = phase_stats.merge(phase_counts, on='rsg')
    
    phase_stats = phase_stats.sort_values('rsg')
    print("Phase Statistics:")
    print(phase_stats.to_string(index=False))
    
    print(f"\nTotal observations by phase (after data cleaning):")
    for _, row in phase_counts.iterrows():
        print(f"  Phase {row['rsg']}: {row['n']} observations")
    print(f"Total: {phase_counts['n'].sum()} observations")

### Signals by Phase

In [ ]:
signals_by_phase = result_by_phase.get_signals()
if not signals_by_phase.empty:
    print(f"Signals Detected: {len(signals_by_phase)} signal(s)\n")
    for i, row in signals_by_phase.iterrows():
        print(f"Signal #{i+1}: {row['chart']} chart in Phase {row['rsg']}")
else:
    print("✓ No signals detected - Process stable across phases")

In [ ]:
signals_by_phase = result_by_phase.get_signals()
print("\n" + "="*60)
print("PHASE ANALYSIS SUMMARY")
print("="*60)
print(f"Phases analyzed: {phase_stats['rsg'].nunique()}")
print(f"Observations per phase: {phase_stats['n'].mean():.0f} (average)")
print(f"Overall mean fillweight: {phase_stats['xbar'].mean():.3f}")
print(f"Phase-to-phase variation: {phase_stats['xbar'].std():.3f}")
print(f"Signals detected: {len(signals_by_phase)}")

---

# Analysis 3: By Lane × Phase (Full Stratification)

**Question**: How does each lane behave in each phase?

**Strategy**: Group by lane AND phase, track over time

This is the most detailed view - revealing interaction effects and nuanced patterns.

In [10]:
# Analyze by lane × phase
analysis_stratified = pdf.analyze(
    response_var='fill_weight',
    grouping_vars=['lane', 'phase'],
    time_var='pull'
)

result_stratified = analysis_stratified.calculate()

print(f"Detected SDS: {result_stratified.summary['sds']}")
print(f"Available Charts: {list(result_stratified.charts.keys())}")


PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 1: Full Factorial with Complete Replication
   All factor × time cells have n ≥ 2 observations (best case for analysis)
   Replication: Full

📈 Available charts: Xbar, S, Imr
   Selected: Xbar and S Charts (Subgroup Mean and Variation) (recommended)

📋 Data Configuration:
   Response: fill_weight
   Time: pull
   Grouping: lane, phase

✨ Analysis Capabilities:
   • VAS residuals: R1, R2, R3, R4, R5 (R2 method: exact)
   • Main effects: Yes
   • Interactions: Yes


Detected SDS: 1
Available Charts: ['Xbar', 'Sbar']


/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([spec.rsg_var_name], dropna=False)[spec.response_var]
/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:307: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grid_cells = df.groupby([spec.rsg_var_name, spec.time_var], dropna=False).size()
/Users/nicholas/Documents/projects/processbehavior/examples/../processbehavior/sds_detector.py:296: FutureWarning: The default of observed=False is deprecated and will be changed

### Stratified Statistics

Notice how the RSG (Rational Subgroup) labels are naturally sorted despite being combinations:
- Lane 1, Lane 2, Lane 10 (not Lane 1, Lane 10, Lane 2)
- This is the power of the new natural sorting!

In [ ]:
if 'Xbar' in result_stratified.charts:
    # Combine Xbar and Sbar data
    xbar_data = result_stratified.charts['Xbar']['data'][['rsg', 'xbar', 'lcl', 'ucl']].copy()
    sbar_data = result_stratified.charts['Sbar']['data'][['rsg', 's']].copy() if 'Sbar' in result_stratified.charts else None
    
    if sbar_data is not None:
        stratified_stats = xbar_data.merge(sbar_data, on='rsg')
    else:
        stratified_stats = xbar_data
    
    # Add observation counts from ANALYSIS dataset
    strat_counts = result_stratified.dataset.groupby('rsg').size().reset_index(name='n')
    stratified_stats = stratified_stats.merge(strat_counts, on='rsg')
    
    # RSG is already categorical with natural sort order!
    stratified_stats = stratified_stats.sort_values('rsg')
    print(f"Lane × Phase Statistics ({len(stratified_stats)} combinations):")
    print(stratified_stats.to_string(index=False))
    
    print(f"\nObservation distribution:")
    print(f"  Min n: {stratified_stats['n'].min()}")
    print(f"  Max n: {stratified_stats['n'].max()}")
    print(f"  Mean n: {stratified_stats['n'].mean():.1f}")
    print(f"  Total: {stratified_stats['n'].sum()} observations")

### Signals in Stratified Analysis

In [ ]:
signals_stratified = result_stratified.get_signals()
if not signals_stratified.empty:
    print(f"Signals Detected: {len(signals_stratified)} signal(s)\n")
    
    # Group signals by chart type
    for chart_type in signals_stratified['chart'].unique():
        chart_signals = signals_stratified[signals_stratified['chart'] == chart_type]
        print(f"{chart_type} Chart Signals ({len(chart_signals)}):")
        for _, row in chart_signals.iterrows():
            print(f"  - {row['rsg']}: beyond_limits={row['beyond_limits']}")
        print()
else:
    print("✓ No signals detected - All lane×phase combinations stable")

In [ ]:
signals_stratified = result_stratified.get_signals()
print("\n" + "="*60)
print("STRATIFIED ANALYSIS SUMMARY")
print("="*60)
print(f"Combinations analyzed: {stratified_stats['rsg'].nunique()}")
print(f"Observations per combination: {stratified_stats['n'].mean():.0f} (average)")
print(f"Overall mean fillweight: {stratified_stats['xbar'].mean():.3f}")
print(f"Combination-to-combination variation: {stratified_stats['xbar'].std():.3f}")
print(f"Signals detected: {len(signals_stratified)}")

---

# Bonus Analysis: IMR on Stratified Data

**Question**: What if we treat each lane×phase combination as individuals over time?

**Strategy**: IMR (Individual Moving Range) chart

This is useful when you want to track each specific combination as a time series.

In [ ]:
# IMR analysis on lane × phase stratification
analysis_imr = pdf.analyze(
    response_var='fill_weight',
    grouping_vars=['lane', 'phase'],
    time_var='pull',
    analysis_type='Imr'  # Force IMR instead of auto-detection
)

result_imr = analysis_imr.calculate()

print(f"Analysis Type: IMR (Individual Moving Range)")
print(f"Available Charts: {list(result_imr.charts.keys())}")

In [ ]:
if 'Imr' in result_imr.charts:
    imr_data = result_imr.charts['Imr']['data']
    
    # Get statistics by group
    imr_stats = imr_data.groupby('rsg').agg({
        'x': ['count', 'mean', 'std'],
        'mr': 'mean'
    }).round(3)
    
    # Flatten column names
    imr_stats.columns = ['_'.join(col).strip('_') for col in imr_stats.columns]
    
    print("IMR Statistics by Lane×Phase:")
    print(imr_stats)

In [ ]:
signals_imr = result_imr.get_signals()
if not signals_imr.empty:
    print(f"\nIMR Signals: {len(signals_imr)}")
    # Show first 5
    for i, row in signals_imr.head(5).iterrows():
        print(f"  {row['rsg']}: beyond_limits={row['beyond_limits']}")
    if len(signals_imr) > 5:
        print(f"  ... and {len(signals_imr) - 5} more")
else:
    print("✓ No IMR signals detected")

---

# Comparison Across Analysis Strategies

Let's compare what we learned from each approach:

In [ ]:
signals_by_lane = result_by_lane.get_signals()
signals_by_phase = result_by_phase.get_signals()
signals_stratified = result_stratified.get_signals()
signals_imr = result_imr.get_signals()

comparison = pd.DataFrame([
    {
        'Strategy': 'By Lane',
        'Groups': lane_stats['rsg'].nunique() if 'Xbar' in result_by_lane.charts else 'N/A',
        'Avg n': f"{lane_stats['n'].mean():.0f}" if 'Xbar' in result_by_lane.charts else 'N/A',
        'SDS': result_by_lane.summary['sds'],
        'Signals': len(signals_by_lane),
        'Charts': ', '.join(result_by_lane.charts.keys())
    },
    {
        'Strategy': 'By Phase',
        'Groups': phase_stats['rsg'].nunique() if 'Xbar' in result_by_phase.charts else 'N/A',
        'Avg n': f"{phase_stats['n'].mean():.0f}" if 'Xbar' in result_by_phase.charts else 'N/A',
        'SDS': result_by_phase.summary['sds'],
        'Signals': len(signals_by_phase),
        'Charts': ', '.join(result_by_phase.charts.keys())
    },
    {
        'Strategy': 'Lane × Phase (Xbar)',
        'Groups': stratified_stats['rsg'].nunique() if 'Xbar' in result_stratified.charts else 'N/A',
        'Avg n': f"{stratified_stats['n'].mean():.0f}" if 'Xbar' in result_stratified.charts else 'N/A',
        'SDS': result_stratified.summary['sds'],
        'Signals': len(signals_stratified),
        'Charts': ', '.join(result_stratified.charts.keys())
    },
    {
        'Strategy': 'Lane × Phase (IMR)',
        'Groups': len(result_imr.charts['Imr']['data']['rsg'].unique()) if 'Imr' in result_imr.charts else 'N/A',
        'Avg n': 'N/A',
        'SDS': result_imr.summary['sds'],
        'Signals': len(signals_imr),
        'Charts': ', '.join(result_imr.charts.keys())
    }
])

print("\n" + "="*80)
print("ANALYSIS STRATEGY COMPARISON")
print("="*80)
print(comparison.to_string(index=False))

---

# Key Takeaways

## The Power of ProcessBehavior

1. **Flexible Stratification**: Same data, multiple analytical lenses
   - By Lane: Machine comparison
   - By Phase: Temporal analysis
   - Lane × Phase: Full interaction effects

2. **Automatic Intelligence**:
   - SDS detection determines appropriate charts
   - Signal detection runs automatically
   - Type conversion ensures correct sorting

3. **Natural Sorting** (NEW!):
   - Charts display "Lane 1, Lane 2, Lane 10" not "Lane 1, Lane 10, Lane 2"
   - Works with any identifier type (numeric, string, dates)
   - No manual sorting required

4. **Minimal Code, Maximum Insight**:
   - 3 lines per analysis
   - Automatic chart selection
   - Built-in diagnostics

## When to Use Each Strategy

- **By Lane**: When comparing machines/lanes is the primary concern
- **By Phase**: When tracking process changes over time/phases
- **Lane × Phase**: When you need complete visibility into interactions
- **IMR**: When treating each combination as an individual time series

## Next Steps

Try this on your own data! Just replace:
```python
pdf = ProcessDataFrame(your_dataframe)
analysis = pdf.analyze(
    response_var='your_measurement',
    grouping_vars=['your_factors'],
    time_var='your_time_column'
)
```

ProcessBehavior handles the rest!